## Question 8 — Transformer encoder block

A post-normalization encoder block keeps the sequence length and the model width. Multi-head self-attention is added back to the input and layer-normalized. A position-wise ReLU feed-forward network is added to that result and layer-normalized again.

## Code Execution

In [1]:
import numpy as np

**Random source.** Seed 0 makes the parameter draw and the five-token input repeatable.

In [ ]:
RNG = np.random.default_rng(0)

**Attention from Question 6.** The encoder block calls multi-head self-attention. Each head uses stable softmax, and a causal mask is available but unused here.

In [ ]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)


def attention_with_mask(Q, K, V, causal=False):
    """One head: Q (nq, dk), K (nk, dk), V (nk, dv)."""
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    if causal:
        if Q.shape[0] != K.shape[0]:
            raise ValueError("Causal mask requires equal query/key lengths")
        future = np.triu(np.ones(scores.shape, dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights


def multi_head_attention_general(query_states, key_value_states, W_Q, W_K, W_V, W_O, num_heads, causal=False):
    """Packed-head attention: output (nq, d_model), weights (H, nq, nk)."""
    query_states = np.asarray(query_states, dtype=float)
    key_value_states = np.asarray(key_value_states, dtype=float)
    if query_states.ndim != 2 or key_value_states.ndim != 2 or query_states.shape[1] != key_value_states.shape[1]:
        raise ValueError("Input states must be 2D with the same model width")
    d_model = query_states.shape[1]
    if num_heads < 1 or d_model % num_heads:
        raise ValueError("num_heads must divide d_model")
    for W in (W_Q, W_K, W_V, W_O):
        if np.shape(W) != (d_model, d_model):
            raise ValueError("Packed projection matrices must be (d_model, d_model)")
    Q = query_states @ W_Q
    K = key_value_states @ W_K
    V = key_value_states @ W_V
    d_head = d_model // num_heads
    outputs, weights = [], []
    for h in range(num_heads):
        sl = slice(h * d_head, (h + 1) * d_head)
        head_output, head_weights = attention_with_mask(Q[:, sl], K[:, sl], V[:, sl], causal=causal)
        outputs.append(head_output)
        weights.append(head_weights)
    concatenated = np.concatenate(outputs, axis=-1)
    return concatenated @ W_O, np.stack(weights, axis=0)


def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    """Self-attention: output (n, d_model), weights (H, n, n)."""
    return multi_head_attention_general(X, X, W_Q, W_K, W_V, W_O, num_heads)

**Layer norm.** Each token is normalized across its features. $\gamma$ and $\beta$ then rescale and shift that token. $\epsilon$ keeps the standard deviation away from zero.

In [ ]:
def layer_norm(X, gamma, beta, eps=1e-5):
    """Normalize each token over its final feature axis."""
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=-1, keepdims=True)
    var = ((X - mean) ** 2).mean(axis=-1, keepdims=True)
    return (X - mean) / np.sqrt(var + eps) * gamma + beta

**Feed-forward.** The same two linear maps and a ReLU are applied to every token. The hidden width is $d_{ff}$, and the second map returns the width to $d_{model}$.

In [ ]:
def feed_forward(X, W1, b1, W2, b2):
    """(n, d) -> (n, d_ff) -> (n, d), same weights for every token."""
    hidden = np.maximum(0, X @ W1 + b1)
    return hidden @ W2 + b2

**Encoder block.** Add the attention output to $X$ and normalize. Add the feed-forward output to that result and normalize again. The output shape matches $X$.

In [ ]:
def transformer_encoder_block(X, params):
    """Post-normalization encoder block; return (seq_len, d_model)."""
    attn, _ = multi_head_attention(
        X,
        params["W_Q"],
        params["W_K"],
        params["W_V"],
        params["W_O"],
        params["num_heads"],
    )
    H = layer_norm(X + attn, params["gamma1"], params["beta1"])
    ffn = feed_forward(H, params["W1"], params["b1"], params["W2"], params["b2"])
    return layer_norm(H + ffn, params["gamma2"], params["beta2"])

**Random parameters.** The attention and first feed-forward matrices use scale $1/\sqrt{d_{model}}$, and $W_2$ uses $1/\sqrt{d_{ff}}$. Biases start at zero. Layer-norm scales start at one and shifts start at zero.

In [ ]:
def random_encoder_params(d_model=8, d_ff=16, num_heads=2):
    scale = 1 / np.sqrt(d_model)
    return dict(
        W_Q=RNG.normal(0, scale, (d_model, d_model)),
        W_K=RNG.normal(0, scale, (d_model, d_model)),
        W_V=RNG.normal(0, scale, (d_model, d_model)),
        W_O=RNG.normal(0, scale, (d_model, d_model)),
        W1=RNG.normal(0, scale, (d_model, d_ff)),
        b1=np.zeros(d_ff),
        W2=RNG.normal(0, 1 / np.sqrt(d_ff), (d_ff, d_model)),
        b2=np.zeros(d_model),
        gamma1=np.ones(d_model),
        beta1=np.zeros(d_model),
        gamma2=np.ones(d_model),
        beta2=np.zeros(d_model),
        num_heads=num_heads,
    )

**Input.** Five tokens, each of width $8$.

In [ ]:
X8 = RNG.normal(size=(5, 8))

**Check.** The block returns a finite array with the same shape as the input.

In [ ]:
Y8 = transformer_encoder_block(X8, random_encoder_params())
print("Q8 input/output shapes:", X8.shape, Y8.shape)
assert Y8.shape == (5, 8) and np.all(np.isfinite(Y8))
print("Encoder block check passed.")